# Exploratory Data Analysis

In [1]:
import os
import random
import yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                   # non-interactive backend — safe for scripts
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict

In [2]:
#go to project root
os.chdir("../..")                  

In [3]:
#Load config
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
 
RAW_DIR     = Path(cfg["data"]["raw_dir"])   # data/raw/images/
PLOTS_DIR   = Path(cfg["eda"]["plots_dir"]) / "plots"             # notebooks/eda/plots
SEED        = cfg["data"]["seed"]                        # 42
CLASS_MAP   = cfg["classes"]                             # {Plastic:0, Paper:1, ...}
NUM_CLASSES = cfg["data"]["num_classes"]                 # 6
 
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

print("Project root :", os.getcwd())
print("Raw dir      :", RAW_DIR)
print("Plots dir    :", PLOTS_DIR)

Project root : C:\Users\ASUS\Documents\GitHub\Trash-Classification-System
Raw dir      : data\raw\images
Plots dir    : notebooks\eda\plots


In [4]:
# Collect raw records without any class mapping
# just  know: what folders exist, how many images, default vs real_world

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"\nDirectory not found: {RAW_DIR}\n"
        "Unzip the Kaggle dataset so that data/raw/images/<sub-category>/ exists."
    )

raw_records = []
for subfolder in sorted(RAW_DIR.iterdir()):
    if not subfolder.is_dir():
        continue
    for image_type in ["default", "real_world"]:
        type_dir = subfolder / image_type
        if not type_dir.exists():
            continue
        for img_path in type_dir.glob("*.png"):
            raw_records.append({
                "subfolder"  : subfolder.name,
                "image_type" : image_type,
                "filepath"   : str(img_path),
            })

df_raw = pd.DataFrame(raw_records)
print(f"Total images found : {len(df_raw)}")
print(f"Sub-categories     : {df_raw['subfolder'].nunique()}")

Total images found : 15000
Sub-categories     : 30


## Pre Mapping Checks

In [5]:
# 4c. Default vs real_world balance per sub-category
# Each sub-category should have exactly 250 default and 250 real_world
# Any imbalance here means the dataset download was incomplete

split_balance = (
    df_raw.groupby(["subfolder", "image_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Add a flag column
split_balance["balanced"] = (
    (split_balance.get("default", 0) == 250) &
    (split_balance.get("real_world", 0) == 250)
)

print("Default vs real_world balance (expected: 250 each):\n")
print(split_balance.to_string(index=False))

unbalanced = split_balance[~split_balance["balanced"]]
if len(unbalanced) > 0:
    print(f"\nWARNING — {len(unbalanced)} sub-categories are not balanced:")
    print(unbalanced["subfolder"].tolist())
else:
    print("\nAll sub-categories are balanced (250 default + 250 real_world each).")

Default vs real_world balance (expected: 250 each):

                 subfolder  default  real_world  balanced
              aerosol_cans      250         250      True
        aluminum_food_cans      250         250      True
        aluminum_soda_cans      250         250      True
           cardboard_boxes      250         250      True
       cardboard_packaging      250         250      True
                  clothing      250         250      True
            coffee_grounds      250         250      True
disposable_plastic_cutlery      250         250      True
                 eggshells      250         250      True
                food_waste      250         250      True
    glass_beverage_bottles      250         250      True
 glass_cosmetic_containers      250         250      True
           glass_food_jars      250         250      True
                 magazines      250         250      True
                 newspaper      250         250      True
              offic

## Class Mapping

In [6]:
# Keyword → main class mapping 
KEYWORD_TO_CLASS = {
    # Plastic (9 sub-categories)
    "plastic"      : "Plastic",
    "bag"          : "Plastic",
    "straw"        : "Plastic",
    "cup lid"      : "Plastic",
    "cutlery"      : "Plastic",
    "container"    : "Plastic",
    "styrofoam"    : "Plastic",
    # Paper (5 sub-categories)
    "newspaper"    : "Paper",
    "office_paper" : "Paper",
    "office paper" : "Paper",
    "magazine"     : "Paper",
    "cardboard"    : "Paper",
    "paper_cup"    : "Paper",
    # Glass (3 sub-categories)
    "glass"        : "Glass",
    # Metal (4 sub-categories)
    "aluminum"     : "Metal",
    "steel"        : "Metal",
    "aerosol"      : "Metal",
    "metal"        : "Metal",
    # Organic (5 sub-categories)
    "food waste"   : "Organic",
    "food_waste"   : "Organic",
    "eggshell"     : "Organic",
    "coffee"       : "Organic",
    "tea bag"      : "Organic",
    "fruit"        : "Organic",
    "vegetable"    : "Organic",
    # Textile (2 sub-categories)
    "clothing"     : "Textile",
    "shoe"         : "Textile",
}

def get_main_class(subfolder_name: str) -> str:
    name_lower = subfolder_name.lower()
    for keyword, main_class in KEYWORD_TO_CLASS.items():
        if keyword in name_lower:
            return main_class
    return "Unknown"

print("KEYWORD_TO_CLASS defined.")
print(f"Total keywords : {len(KEYWORD_TO_CLASS)}")
print(f"Target classes : {list(CLASS_MAP.keys())}")

KEYWORD_TO_CLASS defined.
Total keywords : 27
Target classes : ['Plastic', 'Paper', 'Glass', 'Metal', 'Organic', 'Textile']


In [7]:
# Step 1: Walk raw directory 
print("Scanning:", RAW_DIR)
 
if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"\nDirectory not found: {RAW_DIR}\n"
        "Unzip the Kaggle dataset so that data/raw/images/<sub-category>/ exists."
    )
 
records = []
for subfolder in sorted(RAW_DIR.iterdir()):
    if not subfolder.is_dir():
        continue
    main_class = get_main_class(subfolder.name)
    for split_type in ["default", "real_world"]:
        split_dir = subfolder / split_type
        if not split_dir.exists():
            continue
        for img_path in split_dir.glob("*.png"):
            records.append({
                "subfolder"  : subfolder.name,
                "main_class" : main_class,
                "split_type" : split_type,
                "filepath"   : str(img_path),
            })
 
df = pd.DataFrame(records)
print(f"Total images found: {len(df)}")
 
unknown = df[df["main_class"] == "Unknown"]
if len(unknown) > 0:
    print(f"\nWARNING — {len(unknown)} images not mapped to any class:")
    print(unknown["subfolder"].unique())
    print("Add the missing keyword to KEYWORD_TO_CLASS in eda.py")
else:
    print("All sub-categories mapped successfully to 6 main classes

SyntaxError: unterminated string literal (detected at line 36) (867389916.py, line 36)

In [ ]:
# Step 2: Console summary 
print("\n" + "=" * 52)
print("  CLASS DISTRIBUTION SUMMARY")
print("=" * 52)
class_counts = df["main_class"].value_counts().reindex(CLASS_MAP.keys(), fill_value=0)
for cls, count in class_counts.items():
    pct = 100 * count / len(df)
    bar = "#" * int(pct / 2)
    print(f"  {cls:<10}  {count:>5}  ({pct:4.1f}%)  {bar}")
print("=" * 52)
 
# Warn if any class is severely imbalanced
print()
mean_count = class_counts.mean()
for cls, count in class_counts.items():
    deviation = abs(count - mean_count) / mean_count
    if deviation > 0.30:
        print(f"WARNING: {cls} has {count} images — {deviation*100:.0f}% off the mean ({mean_count:.0f}).")
        print("         This class imbalance will affect training. Consider class weights.")

# Sub-category breakdown
print("\nSub-category image counts:")
sub_counts = (
    df.groupby(["main_class", "subfolder"])
    .size()
    .reset_index(name="count")
    .sort_values(["main_class", "count"], ascending=[True, False])
)
print(sub_counts.to_string(index=False))

## Plots

In [ ]:
#  Shared colour palette 
CLASS_COLORS = {
    "Plastic" : "#378ADD",
    "Paper"   : "#EF9F27",
    "Glass"   : "#1D9E75",
    "Metal"   : "#888780",
    "Organic" : "#639922",
    "Textile" : "#D4537E",
}
 
def save_fig(fig: plt.Figure, filename: str) -> None:
    path = PLOTS_DIR / filename
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")

In [ ]:
#  Plot 1: Overall 6-class bar chart 
print("\n Plotting class distribution...")
fig, ax = plt.subplots(figsize=(9, 5))
classes = list(class_counts.index)
counts  = list(class_counts.values)
colors  = [CLASS_COLORS[c] for c in classes]
bars    = ax.bar(classes, counts, color=colors, edgecolor="white", linewidth=0.6)
for bar, n in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 40,
        str(n), ha="center", va="bottom", fontsize=10
    )
ax.set_title("Image count per main class", fontsize=13, fontweight="bold", pad=10)
ax.set_ylabel("Number of images")
ax.set_ylim(0, max(counts) * 1.2)
sns.despine(ax=ax)
fig.tight_layout()
save_fig(fig, "class_distribution.png")
 

In [ ]:
# Plot 2: Sub-category distribution (6 subplots, one per class) 
print(" Plotting sub-category distribution...")
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
 
for idx, main_class in enumerate(CLASS_MAP.keys()):
    ax     = axes[idx]
    sub_df = sub_counts[sub_counts["main_class"] == main_class].copy()
 
    # Shorten sub-category names: strip the main class word for readability
    sub_df["short"] = sub_df["subfolder"].str.replace(
        main_class, "", case=False
    ).str.strip().replace("", main_class)
 
    color = CLASS_COLORS[main_class]
    ax.barh(sub_df["short"], sub_df["count"], color=color, edgecolor="white", linewidth=0.5)
    ax.axvline(500, color="gray", linestyle="--", linewidth=0.8, alpha=0.5, label="500 target")
    ax.set_title(main_class, fontsize=11, fontweight="bold")
    ax.set_xlabel("Images")
    ax.legend(fontsize=8)
    sns.despine(ax=ax)
 
fig.suptitle("Sub-category image counts grouped by class", fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
save_fig(fig, "subcategory_distribution.png")
 

In [ ]:
# Plot 3: Sub-category distribution (6 subplots, one per class) 
print(" Plotting sub-category distribution...")
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
 
for idx, main_class in enumerate(CLASS_MAP.keys()):
    ax     = axes[idx]
    sub_df = sub_counts[sub_counts["main_class"] == main_class].copy()
 
    # Shorten sub-category names: strip the main class word for readability
    sub_df["short"] = sub_df["subfolder"].str.replace(
        main_class, "", case=False
    ).str.strip().replace("", main_class)
 
    color = CLASS_COLORS[main_class]
    ax.barh(sub_df["short"], sub_df["count"], color=color, edgecolor="white", linewidth=0.5)
    ax.axvline(500, color="gray", linestyle="--", linewidth=0.8, alpha=0.5, label="500 target")
    ax.set_title(main_class, fontsize=11, fontweight="bold")
    ax.set_xlabel("Images")
    ax.legend(fontsize=8)
    sns.despine(ax=ax)
 
fig.suptitle("Sub-category image counts grouped by class", fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
save_fig(fig, "subcategory_distribution.png")

In [ ]:
#  Plot 4: Default vs real_world 
print(" Plotting default vs real_world...")
pivot = (
    df.groupby(["main_class", "split_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(CLASS_MAP.keys())
)
 
fig, ax = plt.subplots(figsize=(10, 5))
x  = np.arange(len(pivot))
w  = 0.35
ax.bar(x - w/2, pivot.get("default",    pd.Series([0]*len(pivot))),
       width=w, label="default (studio)",    color="#378ADD", alpha=0.85)
ax.bar(x + w/2, pivot.get("real_world", pd.Series([0]*len(pivot))),
       width=w, label="real_world (in-use)", color="#D85A30", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=12)
ax.set_title("Default vs real-world counts per class", fontsize=13, fontweight="bold", pad=10)
ax.set_ylabel("Number of images")
ax.legend()
sns.despine(ax=ax)
fig.tight_layout()
save_fig(fig, "default_vs_realworld.png")

In [ ]:
#  Plot 5: Sample image grid (3 rows × 6 cols)
print(" Building sample image grid...")
N_SAMPLES = 3
fig = plt.figure(figsize=(14, 7))
fig.suptitle(
    "Sample images per class (default + real_world mixed)",
    fontsize=13, fontweight="bold"
)
gs = gridspec.GridSpec(N_SAMPLES, NUM_CLASSES, figure=fig, hspace=0.3, wspace=0.05)
 
for col, main_class in enumerate(CLASS_MAP.keys()):
    class_df = df[df["main_class"] == main_class]
    samples  = class_df.sample(min(N_SAMPLES, len(class_df)), random_state=SEED)
    for row, (_, record) in enumerate(samples.iterrows()):
        ax = fig.add_subplot(gs[row, col])
        try:
            img = Image.open(record["filepath"]).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "load\nerror", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8)
        ax.axis("off")
        if row == 0:
            ax.set_title(
                main_class, fontsize=9, fontweight="bold",
                color=CLASS_COLORS[main_class], pad=4
            )
 
save_fig(fig, "sample_grid.png")

In [ ]:
# Plot 6: Pixel brightness and std per class 
print(" Computing pixel statistics (sampling 100 images per class)...")
brightness_per_class = defaultdict(list)
std_per_class        = defaultdict(list)
 
for main_class in CLASS_MAP.keys():
    class_df = df[df["main_class"] == main_class]
    sample   = class_df.sample(min(100, len(class_df)), random_state=SEED)
    for _, row in sample.iterrows():
        try:
            arr = np.array(
                Image.open(row["filepath"]).convert("RGB").resize((64, 64))
            ) / 255.0
            brightness_per_class[main_class].append(arr.mean())
            std_per_class[main_class].append(arr.std())
        except Exception:
            continue
 
mean_brightness = {c: float(np.mean(v)) for c, v in brightness_per_class.items()}
mean_std        = {c: float(np.mean(v)) for c, v in std_per_class.items()}
 
print("\n  Pixel stats summary:")
print(f"  {'Class':<10}  {'Mean brightness':>16}  {'Mean std':>10}")
for c in CLASS_MAP.keys():
    print(f"  {c:<10}  {mean_brightness[c]:>16.4f}  {mean_std[c]:>10.4f}")
 
classes_ordered = list(CLASS_MAP.keys())
colors_ordered  = [CLASS_COLORS[c] for c in classes_ordered]
 
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
axes[0].bar(classes_ordered, [mean_brightness[c] for c in classes_ordered],
            color=colors_ordered, edgecolor="white")
axes[0].set_title("Mean pixel brightness per class", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Mean brightness  (0 = black, 1 = white)")
axes[0].set_ylim(0, 1)
axes[0].set_xticklabels(classes_ordered, rotation=12)
sns.despine(ax=axes[0])
 
axes[1].bar(classes_ordered, [mean_std[c] for c in classes_ordered],
            color=colors_ordered, edgecolor="white")
axes[1].set_title("Mean pixel std deviation per class", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Std deviation  (higher = more varied colours)")
axes[1].set_ylim(0, 0.5)
axes[1].set_xticklabels(classes_ordered, rotation=12)
sns.despine(ax=axes[1])
 
fig.suptitle(
    "Pixel statistics — sample of 100 images per class",
    fontsize=13, fontweight="bold"
)
fig.tight_layout()
save_fig(fig, "pixel_stats.png")

print("\nEDA complete. All 5 plots saved to:", PLOTS_DIR)
print("Share notebooks/eda/ with the team, then Member 1 runs preprocessing.ipynb.")
 

In [ ]:
# Plot 7: Image dimensions per class
print(" Image dimensions per class")
widths_per_class  = defaultdict(list)
heights_per_class = defaultdict(list)

for main_class in CLASS_MAP.keys():
    class_df = df[df["main_class"] == main_class]
    sample   = class_df.sample(min(100, len(class_df)), random_state=SEED)
    for _, row in sample.iterrows():
        try:
            img = Image.open(row["filepath"])
            w, h = img.size          # PIL gives (width, height)
            widths_per_class[main_class].append(w)
            heights_per_class[main_class].append(h)
        except Exception:
            continue

# --- Console summary ---
print(f"\n  {'Class':<10}  {'Avg Width':>10}  {'Avg Height':>11}  {'Min W':>6}  {'Max W':>6}  {'Min H':>6}  {'Max H':>6}")
for c in CLASS_MAP.keys():
    ws = widths_per_class[c]
    hs = heights_per_class[c]
    print(f"  {c:<10}  {np.mean(ws):>10.1f}  {np.mean(hs):>11.1f}  "
          f"{min(ws):>6}  {max(ws):>6}  {min(hs):>6}  {max(hs):>6}")

In [ ]:
# Plot 8: Aspect Ratio Distribution per Class
print(" Aspect Ratio Distribution per Class")
aspect_ratios_per_class = defaultdict(list)
aspect_labels = []   # for the full scatter/violin data

for main_class in CLASS_MAP.keys():
    class_df = df[df["main_class"] == main_class]
    sample   = class_df.sample(min(100, len(class_df)), random_state=SEED)
    for _, row in sample.iterrows():
        try:
            img = Image.open(row["filepath"])
            w, h = img.size
            ratio = w / h
            aspect_ratios_per_class[main_class].append(ratio)
        except Exception:
            continue

# --- Console Summary ---
print(f"\n  {'Class':<10}  {'Mean AR':>8}  {'Std AR':>7}  {'Min AR':>7}  {'Max AR':>7}  {'% Square (0.9–1.1)':>20}")
for c in CLASS_MAP.keys():
    ratios = aspect_ratios_per_class[c]
    pct_square = 100 * sum(0.9 <= r <= 1.1 for r in ratios) / len(ratios)
    print(f"  {c:<10}  {np.mean(ratios):>8.3f}  {np.std(ratios):>7.3f}  "
          f"{min(ratios):>7.3f}  {max(ratios):>7.3f}  {pct_square:>20.1f}%")

# --- Reference lines for target resolutions ---
# EfficientNet-B3 → 300x300 (AR = 1.0)
# ResNet-50, MobileNetV3, Swin-T → 224x224 (AR = 1.0)
# All 4 models expect square input, so AR = 1.0 is the target

classes_ordered = list(CLASS_MAP.keys())

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Plot A: Violin plot of aspect ratios per class ---
violin_data = [aspect_ratios_per_class[c] for c in classes_ordered]
parts = axes[0].violinplot(violin_data, positions=range(len(classes_ordered)),
                            showmedians=True, showextrema=True)

# Color each violin
for i, (pc, c) in enumerate(zip(parts["bodies"], classes_ordered)):
    pc.set_facecolor(CLASS_COLORS[c])
    pc.set_alpha(0.75)
parts["cmedians"].set_color("black")
parts["cmedians"].set_linewidth(2)

axes[0].axhline(1.0, color="red", linestyle="--", linewidth=1.2, label="Square (AR=1.0)")
axes[0].axhline(0.9, color="orange", linestyle=":", linewidth=1.0, label="±10% band")
axes[0].axhline(1.1, color="orange", linestyle=":", linewidth=1.0)
axes[0].set_xticks(range(len(classes_ordered)))
axes[0].set_xticklabels(classes_ordered, rotation=15)
axes[0].set_title("Aspect ratio distribution per class\n(violin)", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Aspect Ratio  (width / height)")
axes[0].legend(fontsize=8)
sns.despine(ax=axes[0])

# --- Plot B: Box plot of aspect ratios per class ---
bp = axes[1].boxplot(violin_data, labels=classes_ordered, patch_artist=True,
                      medianprops=dict(color="black", linewidth=2))
for patch, c in zip(bp["boxes"], classes_ordered):
    patch.set_facecolor(CLASS_COLORS[c])
    patch.set_alpha(0.75)

axes[1].axhline(1.0, color="red", linestyle="--", linewidth=1.2, label="Square (AR=1.0)")
axes[1].axhline(0.9, color="orange", linestyle=":", linewidth=1.0, label="±10% band")
axes[1].axhline(1.1, color="orange", linestyle=":", linewidth=1.0)
axes[1].set_title("Aspect ratio distribution per class\n(box plot)", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Aspect Ratio  (width / height)")
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend(fontsize=8)
sns.despine(ax=axes[1])

# --- Plot C: Histogram of AR across ALL images, stacked by class ---
bins = np.linspace(0.4, 2.0, 40)
for c in classes_ordered:
    axes[2].hist(
        aspect_ratios_per_class[c], bins=bins,
        color=CLASS_COLORS[c], alpha=0.55, label=c, edgecolor="none"
    )
axes[2].axvline(1.0, color="red", linestyle="--", linewidth=1.5, label="Square (AR=1.0)")
axes[2].axvline(0.9, color="orange", linestyle=":", linewidth=1.0)
axes[2].axvline(1.1, color="orange", linestyle=":", linewidth=1.0)
axes[2].set_title("Aspect ratio histogram\n(all classes overlaid)", fontsize=11, fontweight="bold")
axes[2].set_xlabel("Aspect Ratio  (width / height)")
axes[2].set_ylabel("Image count")
axes[2].legend(fontsize=8)
sns.despine(ax=axes[2])

fig.suptitle(
    "Aspect ratio analysis — sample of 100 images per class\n"
    "Red dashed = square target (all 4 models expect 1:1 input)",
    fontsize=13, fontweight="bold"
)
fig.tight_layout()
save_fig(fig, "aspect_ratio_distribution.png")